# Doctor Models: SHAP Explainability (all three models)

Run All. The notebook loads the **discharge**, **transfer** and **stay** models automatically and explains them with `shap.LinearExplainer`. No settings to change.

**Integration functions**
* `explain(model_key, record)`: one model, returns a JSON-ready dict
* `explain_all(record)`: all three models for one patient
* every result includes `doctor_text`, a plain-language summary for the doctor's screen

**Required files**
```
artifacts/discharge_outcome_model.joblib   artifacts/discharge_outcome_metadata.joblib
artifacts/transfer_model.joblib            artifacts/transfer_metadata.joblib
artifacts/stay_duration_model.joblib       artifacts/stay_duration_metadata.joblib
Preprocessed_Dataset.csv   (only needed the first time; SHAP backgrounds are then cached in artifacts/)
```

## 1. Explainer code

In [ ]:
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import shap
from sklearn.linear_model import LogisticRegression

logger = logging.getLogger("doctor_shap")

MODEL_DIR = Path(os.getenv("MODEL_DIR", "artifacts"))
DATA_PATH = Path(os.getenv("DATA_PATH", "Preprocessed_Dataset.csv"))
BACKGROUND_SIZE = int(os.getenv("BACKGROUND_SIZE", "300"))
TOP_K = int(os.getenv("TOP_K", "5"))
RANDOM_STATE = 42

MODEL_FILES = {
    "discharge": ("discharge_outcome_model.joblib", "discharge_outcome_metadata.joblib"),
    "transfer": ("transfer_model.joblib", "transfer_metadata.joblib"),
    "stay": ("stay_duration_model.joblib", "stay_duration_metadata.joblib"),
}
MODEL_TITLES = {
    "discharge": "Discharge outcome",
    "transfer": "Transfer to another hospital",
    "stay": "Expected hospital stay",
}
CLASS_TEXT = {("transfer", "yes"): "Transfer likely", ("transfer", "no"): "Transfer unlikely"}
OUTCOME_TEXT = {("transfer", "yes"): "transfer", ("transfer", "no"): "no transfer"}

DISPLAY_NAMES = {
    "Age": "Age",
    "Gender": "Gender",
    "Mode of Transport to the Hospital": "How they reached hospital",
    "Time Taken To Reach Hospital": "Time taken to reach hospital",
    "Vehicle Type": "Vehicle type",
    "Helmet Worn": "Helmet worn",
    "Collision With": "What they collided with",
    "Collision Force From": "Direction of impact",
    "Approximate Speed": "Approximate speed",
    "Road Type": "Road type",
    "Road Condition": "Road condition",
    "Alcohol Consumption": "Alcohol consumption",
    "Site of Injury No1": "Main injury site",
    "Type of injury No 1": "Main injury type",
    "Site of injury No 2": "Second injury site",
    "Type of Injury No 2": "Second injury type",
    "Investigation Done": "Investigation done",
    "Management Name 1": "First treatment",
    "Management Name 2": "Second treatment",
    "Transfer To Next Hospital": "Transferred to next hospital",
    "State of Transfer": "State on transfer",
    "total_days": "Days in hospital so far",
}

STRONG, MODERATE, SLIGHT = 0.5, 0.2, 0.05     # effect size thresholds (log-odds)
LOW_CONFIDENCE = 0.5
DISCLAIMER = "Decision support only. The final clinical decision rests with the doctor."


class ExplainError(Exception):
    """Raised for problems that should surface as a clean API error."""


def _normalise_text(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for c in out.columns:
        if out[c].dtype == object or pd.api.types.is_string_dtype(out[c]):
            out[c] = (out[c].astype(str).str.strip().str.lower()
                      .replace({"nan": np.nan, "none": np.nan, "": np.nan}))
    return out


def _dense(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


def _strength(v: float) -> str:
    a = abs(v)
    return "strong" if a >= STRONG else "moderate" if a >= MODERATE else "slight"


def load_patient_table(path: Path = DATA_PATH) -> pd.DataFrame:
    """Training data prepared the same way as in the model notebooks."""
    df = pd.read_csv(path, low_memory=False).rename(columns={"Collision With ": "Collision With"})
    df = _normalise_text(df)
    ward = ["Number of days in first ward", "Number of days in Second ward",
            "Number of days in Third ward"]
    for c in ward:
        df[c] = pd.to_numeric(df[c], errors="coerce")
        reason = f"{c}_missing_reason"
        if reason in df.columns:
            early = df[reason] == "early_discharge"
            df.loc[early, c] = df.loc[early, c].fillna(0)
    df["total_days"] = df[ward].sum(axis=1, min_count=1)
    return df


class DoctorExplainer:
    """Explainer for one doctor model."""

    def __init__(self, model_key: str, model_dir: Path = MODEL_DIR, patients: pd.DataFrame | None = None):
        if model_key not in MODEL_FILES:
            raise ExplainError(f"Unknown model '{model_key}'. Valid: {list(MODEL_FILES)}")
        self.key = model_key
        model_file, meta_file = MODEL_FILES[model_key]
        mp, tp = Path(model_dir) / model_file, Path(model_dir) / meta_file
        if not mp.exists() or not tp.exists():
            raise ExplainError(f"Missing files for '{model_key}': {mp} and {tp}")
        self.model = joblib.load(mp)
        self.meta = joblib.load(tp)
        self.features = list(self.meta["features"])

        steps = getattr(self.model, "named_steps", {})
        if "pre" not in steps or not isinstance(steps.get("clf"), LogisticRegression):
            raise ExplainError(f"'{model_key}' is not a 'pre' + LogisticRegression pipeline")
        self.pre, self.clf = steps["pre"], steps["clf"]
        self.classes = list(self.clf.classes_)

        background = self._load_background(Path(model_dir), patients)
        self.background = background
        self.explainer = shap.LinearExplainer(
            self.clf, shap.maskers.Independent(background, max_samples=len(background)))

        by_len = sorted(self.features, key=len, reverse=True)
        self.origin = []
        for name in self.pre.get_feature_names_out():
            base = name.split("__", 1)[-1]
            self.origin.append(next((f for f in by_len if base == f or base.startswith(f + "_")), base))
        logger.info("Ready: %s (%d features, classes %s)", model_key, len(self.features), self.classes)

    # ---------- data preparation ----------
    def prepare(self, frame: pd.DataFrame) -> pd.DataFrame:
        out = _normalise_text(frame.reindex(columns=self.features))
        for col, keep in (self.meta.get("rare_maps") or {}).items():
            if col in out.columns:
                out[col] = out[col].where(out[col].isna() | out[col].isin(set(keep)), "other")
        for col, order in (("Age", self.meta["age_order"]),
                           ("Time Taken To Reach Hospital", self.meta["time_order"])):
            if col in out.columns:
                out[col] = out[col].map({v: i for i, v in enumerate(order)})
        return out

    def _load_background(self, model_dir: Path, patients: pd.DataFrame | None) -> np.ndarray:
        """Use a cached background if present, otherwise build one from the data and cache it."""
        cache = model_dir / f"{self.key}_shap_background.joblib"
        if cache.exists():
            return joblib.load(cache)
        if patients is None:
            if not DATA_PATH.exists():
                raise ExplainError(f"No SHAP background for '{self.key}': add {cache.name} "
                                   f"or {DATA_PATH}")
            patients = load_patient_table(DATA_PATH)
        missing = [f for f in self.features if f not in patients.columns]
        if missing:
            raise ExplainError(f"'{self.key}' expects columns not in the data: {missing}")
        sample = patients[self.features].sample(min(BACKGROUND_SIZE, len(patients)),
                                                random_state=RANDOM_STATE)
        background = _dense(self.pre.transform(self.prepare(sample)))
        try:
            joblib.dump(background, cache)
        except OSError:
            logger.warning("Could not cache background for %s", self.key)
        return background

    # ---------- explanation ----------
    def _shap_for_class(self, values, idx: int) -> np.ndarray:
        if isinstance(values, list):
            return np.asarray(values[idx])[0]
        v = np.asarray(values)
        if v.ndim == 3:
            return v[0, :, idx]
        return v[0] if idx == 1 or len(self.classes) > 2 else -v[0]

    def explain(self, record: dict, top_k: int = TOP_K) -> dict:
        raw = _normalise_text(pd.DataFrame([{f: record.get(f) for f in self.features}]))
        x = _dense(self.pre.transform(self.prepare(raw)))
        proba = self.clf.predict_proba(x)[0]
        idx = int(np.argmax(proba))
        predicted, prob = str(self.classes[idx]), float(proba[idx])

        sv = self._shap_for_class(self.explainer.shap_values(x), idx)
        per_feature = pd.Series(sv, index=self.origin).groupby(level=0).sum()
        ranked = per_feature.reindex(per_feature.abs().sort_values(ascending=False).index)

        row = raw.iloc[0]
        recorded = [f for f in ranked.index if pd.notna(row[f])]
        shown = [(f, float(ranked[f])) for f in recorded if abs(ranked[f]) >= SLIGHT][:top_k]

        def item(f, v):
            return {"factor": DISPLAY_NAMES.get(f, f), "value": str(row[f]),
                    "effect": _strength(v), "shap": round(v, 4)}

        note = None
        if not recorded:
            note = "No details recorded yet, so this reflects a typical patient."
        elif prob < LOW_CONFIDENCE:
            note = "Low confidence: the model is unsure, so treat this as a weak signal."

        result = {
            "model": self.key,
            "title": MODEL_TITLES[self.key],
            "prediction": predicted,
            "headline": f"{CLASS_TEXT.get((self.key, predicted), predicted.capitalize())} "
                        f"({prob:.0%} probability)",
            "outcome": OUTCOME_TEXT.get((self.key, predicted), predicted.lower()),
            "probabilities": {str(c): round(float(p), 4) for c, p in zip(self.classes, proba)},
            "points_towards": [item(f, v) for f, v in shown if v > 0],
            "points_away": [item(f, v) for f, v in shown if v < 0],
            "not_recorded": [DISPLAY_NAMES.get(f, f) for f in ranked.index if pd.isna(row[f])][:4],
            "note": note,
            "disclaimer": DISCLAIMER,
        }
        result["doctor_text"] = doctor_text(result)
        return result

    def global_importance(self) -> dict:
        """Average |SHAP| per feature over the background patients."""
        sv = self.explainer.shap_values(self.background)
        arr = np.abs(np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv))
        arr = arr.mean(-1) if arr.ndim == 3 else arr
        imp = pd.Series(arr.mean(0), index=self.origin).groupby(level=0).sum().sort_values(ascending=False)
        return {DISPLAY_NAMES.get(k, k): round(float(v), 4) for k, v in imp.items()}


def doctor_text(r: dict) -> str:
    lines = [f"{r['title']}: {r['headline']}"]
    if r["note"]:
        lines.append(f"  ! {r['note']}")
    if r["points_towards"]:
        lines.append(f"  Points towards {r['outcome']}:")
        lines += [f"    + {x['factor']}: {x['value']} ({x['effect']})" for x in r["points_towards"]]
    if r["points_away"]:
        lines.append(f"  Points away from {r['outcome']}:")
        lines += [f"    - {x['factor']}: {x['value']} ({x['effect']})" for x in r["points_away"]]
    if r["not_recorded"]:
        lines.append("  Not yet recorded: " + ", ".join(r["not_recorded"]))
    lines.append("  " + r["disclaimer"])
    return "\n".join(lines)


def load_all(model_dir: Path = MODEL_DIR) -> dict[str, DoctorExplainer]:
    """Load explainers for all three doctor models (reads the data at most once)."""
    patients = None
    need_data = any(not (Path(model_dir) / f"{k}_shap_background.joblib").exists() for k in MODEL_FILES)
    if need_data and DATA_PATH.exists():
        patients = load_patient_table(DATA_PATH)
    return {k: DoctorExplainer(k, model_dir, patients) for k in MODEL_FILES}


## 2. Load all three models

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

EXPLAINERS = load_all()


def explain(model_key, record):
    if model_key not in EXPLAINERS:
        raise ExplainError(f"Unknown model '{model_key}'. Valid: {list(EXPLAINERS)}")
    return EXPLAINERS[model_key].explain(record)


def explain_all(record):
    return {k: e.explain(record) for k, e in EXPLAINERS.items()}


print("Loaded:", list(EXPLAINERS))

## 3. Example: one patient, all three models

In [ ]:
import json
patients = load_patient_table(DATA_PATH) if DATA_PATH.exists() else None
sample = patients.iloc[7].to_dict() if patients is not None else {
    "Age": "60-69", "Vehicle Type": "motor cyclist", "Helmet Worn": "no",
    "Type of injury No 1": "fracture", "Site of Injury No1": "leg tibia",
    "Mode of Transport to the Hospital": "ambulance", "Collision With": "fall from vehicle"}

for result in explain_all(sample).values():
    print(result["doctor_text"], end="\n\n")

In [ ]:
# JSON returned to the front end (one model shown)
print(json.dumps(explain("transfer", sample), indent=2))

## 4. Check: explanations match each model exactly

SHAP values plus the base value must equal the model's own output (log-odds). Errors near zero confirm the explanation is exact.

In [ ]:
import numpy as np
for key, e in EXPLAINERS.items():
    X = e.background[:100]
    margins = e.clf.decision_function(X)
    base = np.atleast_1d(e.explainer.expected_value)
    sv = e.explainer.shap_values(X)
    if margins.ndim == 1:
        s = np.asarray(sv[0] if isinstance(sv, list) else sv)
        err = np.abs(base[0] + s.sum(1) - margins).max()
    else:
        arr = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)
        err = np.abs(base + arr.sum(1) - margins).max()
    print(f"{key:<10} max error: {err:.2e}")

## 5. Which features matter most overall

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (key, e) in zip(axes, EXPLAINERS.items()):
    imp = pd.Series(e.global_importance()).head(8)[::-1]
    ax.barh(imp.index, imp.values, color="#2c5f8d")
    ax.set_title(MODEL_TITLES[key])
    ax.set_xlabel("Mean |SHAP| (log-odds)")
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("shap_global_all_models.png", dpi=150)
plt.show()

## 6. Integration

Load once at startup and call per request:

```python
EXPLAINERS = load_all()                         # section 2

@router.post("/api/doctor/explain")
def explain_patient(record: dict, user=Depends(require_role("doctor"))):
    return explain_all(record)                  # all three models

@router.post("/api/doctor/explain/{model_key}")
def explain_one(model_key: str, record: dict, user=Depends(require_role("doctor"))):
    return explain(model_key, record)
```

To use it as a Python module: `jupyter nbconvert --to script Doctor_SHAP_Explainability.ipynb`, keeping sections 1 and 2.